# Pharmacon — Writer W02 (Aryan) Adaptation & Clinical Benchmark

This notebook executes the end-to-end evaluation pipeline for **Writer W02**:
1. **Unpack Package & Install Dependencies**
2. **Auto-crop & Perspective Warp** for all 7 sheets
3. **Strict Protocol Partitioning**: 16 Train (`E01-E16`), 4 Val (`V01-V04`), 8 Sealed Test (`T01-T08`)
4. **Generic Baseline Evaluation** on `T01-T08`
5. **LoRA Fine-Tuning** with Early Stopping on `V01-V04`
6. **Adapted Evaluation** on `T01-T08` and Side-by-Side CER/WER Comparison
7. **Clinical Entity Resolution** via FormularyMatcher (NLEM 2022)


In [ ]:
# Cell 1: Install dependencies and unpack package
!pip install -q "sentencepiece>=0.2.0" "protobuf>=3.20.3" "transformers>=4.40.0" "peft>=0.10.0" "accelerate>=0.28.0" "pymupdf>=1.24.0" "rapidfuzz>=3.0.0"
!unzip -q -o /content/pharmacon_w02_run.zip -d /content/
print("Package extracted and dependencies installed.")


In [ ]:
# Cell 2: Master Execution Pipeline (Writer W02)
import os, sys, copy, cv2, torch
import numpy as np
from transformers import AutoImageProcessor, XLMRobertaTokenizer, TrOCRProcessor, VisionEncoderDecoderModel
from peft import LoraConfig, get_peft_model
from pharmacon_ai import process_photo, evaluate_condition, print_comparison
from pharmacon_ai.formulary import FormularyMatcher

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

WRITER_ID = "W02"
IMAGES_DIR = "/content"
PDF_PATH   = "/content/Pharmacon_Writer_Calibration_Protocol.pdf"

PAGE_PHOTOS = {
    2: os.path.join(IMAGES_DIR, "W2TS1.jpg"),       # E01-E04
    3: os.path.join(IMAGES_DIR, "W2TS2.jpg"),       # E05-E08
    4: os.path.join(IMAGES_DIR, "W2TS3.jpg"),       # E09-E12
    5: os.path.join(IMAGES_DIR, "W2TS4.jpg"),       # E13-E16
    6: os.path.join(IMAGES_DIR, "W2VS.jpg"),        # V01-V04, R01-R04
    7: os.path.join(IMAGES_DIR, "W2HS1.jpg"),       # T01-T04
    8: os.path.join(IMAGES_DIR, "W2HS2.jpg"),       # T05-T08
}

# 1. EXTRACT CROPS
all_lines = {}
for p_idx, p_path in PAGE_PHOTOS.items():
    all_lines.update(process_photo(p_path, PDF_PATH, p_idx, writer_id=WRITER_ID))
print(f"Extracted: {len(all_lines)} / 28 lines")

# 2. STRICT PROTOCOL SPLITS
train_ids = [f"E{i:02d}" for i in range(1, 17)]
val_ids   = [f"V{i:02d}" for i in range(1, 5)]
test_ids  = [f"T{i:02d}" for i in range(1, 9)]

train_crops  = [cv2.cvtColor(all_lines[lid]["image"], cv2.COLOR_BGR2RGB) for lid in train_ids]
train_labels = [all_lines[lid]["label"] for lid in train_ids]

val_crops    = [cv2.cvtColor(all_lines[lid]["image"], cv2.COLOR_BGR2RGB) for lid in val_ids]
val_labels   = [all_lines[lid]["label"] for lid in val_ids]

test_crops   = {lid: cv2.cvtColor(all_lines[lid]["image"], cv2.COLOR_BGR2RGB) for lid in test_ids}
test_labels  = {lid: all_lines[lid]["label"] for lid in test_ids}

del all_lines  # Sealed guarantee
print(f"Data ready: {len(train_crops)} Train | {len(val_crops)} Val | {len(test_crops)} Sealed Test")

# 3. BASELINE MODEL
MODEL_ID = "microsoft/trocr-small-handwritten"
image_processor = AutoImageProcessor.from_pretrained(MODEL_ID)
tokenizer       = XLMRobertaTokenizer.from_pretrained(MODEL_ID)
processor       = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)

model = VisionEncoderDecoderModel.from_pretrained(MODEL_ID).to(device)
model.config.decoder_start_token_id = processor.tokenizer.eos_token_id
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.vocab_size             = model.config.decoder.vocab_size

# 4. RUN GENERIC BASELINE
print(f"\n--- Generic Baseline on Writer {WRITER_ID} (T01-T08) ---")
model.eval()
generic_preds = {}
with torch.no_grad():
    for lid, img in test_crops.items():
        pv = processor(images=img, return_tensors="pt").pixel_values.to(device)
        ids = model.generate(pv, max_new_tokens=64, num_beams=4, early_stopping=True, no_repeat_ngram_size=3, repetition_penalty=1.3)
        generic_preds[lid] = processor.batch_decode(ids, skip_special_tokens=True)[0].strip()

generic_results = evaluate_condition(generic_preds, test_labels)
print(f"Generic MACRO: CER={generic_results['macro']['cer']:.4f} | WER={generic_results['macro']['wer']:.4f}")

# 5. LORA ADAPTER
try:
    import peft.import_utils; peft.import_utils.is_torchao_available = lambda: False
    import peft.tuners.lora.torchao; peft.tuners.lora.torchao.is_torchao_available = lambda: False
except Exception:
    pass

targets = [
    "encoder.layers.10.attention.q_proj", "encoder.layers.10.attention.v_proj",
    "encoder.layers.11.attention.q_proj", "encoder.layers.11.attention.v_proj"
]
lora_config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.15, target_modules=targets, bias="none")
peft_model = get_peft_model(model, lora_config)

# 6. TRAIN ON E01-E16 WITH VAL ON V01-V04
opt = torch.optim.AdamW([p for p in peft_model.parameters() if p.requires_grad], lr=1e-4, weight_decay=0.01)
best_val_loss, best_weights, patience, p_count = float("inf"), None, 4, 0

print(f"\n--- Training Writer {WRITER_ID} LoRA ---")
for epoch in range(1, 41):
    peft_model.train()
    t_loss = 0.0
    for img, text in zip(train_crops, train_labels):
        pv = processor(images=img, return_tensors="pt").pixel_values.to(device)
        lbl = processor.tokenizer(text=text, return_tensors="pt").input_ids.to(device)
        loss = peft_model(pixel_values=pv, labels=lbl).loss
        opt.zero_grad(); loss.backward(); opt.step()
        t_loss += loss.item()
    t_loss /= len(train_crops)

    peft_model.eval()
    v_loss = 0.0
    with torch.no_grad():
        for img, text in zip(val_crops, val_labels):
            pv = processor(images=img, return_tensors="pt").pixel_values.to(device)
            lbl = processor.tokenizer(text=text, return_tensors="pt").input_ids.to(device)
            v_loss += peft_model(pixel_values=pv, labels=lbl).loss.item()
    v_loss /= len(val_crops)

    if epoch % 5 == 0 or v_loss < best_val_loss:
        print(f"Epoch {epoch:02d} | Train: {t_loss:.4f} | Val: {v_loss:.4f}")

    if v_loss < best_val_loss:
        best_val_loss = v_loss
        best_weights = copy.deepcopy(peft_model.state_dict())
        p_count = 0
    else:
        p_count += 1
        if p_count >= patience:
            print(f"Early stopping at epoch {epoch}. Best Val Loss: {best_val_loss:.4f}")
            break

if best_weights:
    peft_model.load_state_dict(best_weights)

# 7. EVALUATE ADAPTED MODEL
print(f"\n--- Running Adapted Model on Writer {WRITER_ID} (T01-T08) ---")
peft_model.eval()
adapted_preds = {}
with torch.no_grad():
    for lid, img in test_crops.items():
        pv = processor(images=img, return_tensors="pt").pixel_values.to(device)
        ids = peft_model.generate(pv, max_new_tokens=64, num_beams=4, early_stopping=True, no_repeat_ngram_size=3, repetition_penalty=1.3)
        adapted_preds[lid] = processor.batch_decode(ids, skip_special_tokens=True)[0].strip()

adapted_results = evaluate_condition(adapted_preds, test_labels)

# 8. PRINT BENCHMARK COMPARISON TABLE
print("\n" + "="*80)
print(f"PHARMACON BENCHMARK — Writer {WRITER_ID} (Generic vs Adapted on Sealed T01-T08)")
print("="*80)
print_comparison(generic_results, adapted_results, label_a="Generic", label_b="Adapted")

# 9. CLINICAL RESOLUTION (FORMULARY MATCHING)
print("\n" + "="*80)
print(f"CLINICAL RESOLUTION (Formulary Matching on Writer {WRITER_ID})")
print("="*80)
matcher = FormularyMatcher()
for lid in sorted(test_labels.keys()):
    raw = adapted_preds[lid]
    target = test_labels[lid]
    res = matcher.resolve(raw)
    top = res["top_candidates"][0] if res["top_candidates"] else {}
    ent = res["parsed_entities"]
    drug = top.get("generic_name", "UNKNOWN")
    conf = top.get("confidence", 0.0)
    gate = res["safety_gate"]["status"]
    strength = ent.get("strength") or ""
    form = ent.get("form") or ""
    resolved_str = f"{drug} {strength} {form}".strip()
    print(f"[{lid}] Target   : {target}")
    print(f"     Raw OCR  : {raw}")
    print(f"     Resolved : {resolved_str} (Conf: {conf:.2f}, Gate: {gate})")


In [ ]:
# Cell 3: Formulary Resolution Standalone (Instant - No retraining needed)
from pharmacon_ai.formulary import FormularyMatcher
matcher = FormularyMatcher()

print("="*80)
print("CLINICAL RESOLUTION (Formulary Matching on Writer W02)")
print("="*80)
for lid in sorted(test_labels.keys()):
    raw = adapted_preds[lid]
    target = test_labels[lid]
    res = matcher.resolve(raw)
    top = res["top_candidates"][0] if res["top_candidates"] else {}
    ent = res["parsed_entities"]
    drug = top.get("generic_name", "UNKNOWN")
    conf = top.get("confidence", 0.0)
    gate = res["safety_gate"]["status"]
    strength = ent.get("strength") or ""
    form = ent.get("form") or ""
    resolved_str = f"{drug} {strength} {form}".strip()
    print(f"[{lid}] Target   : {target}")
    print(f"     Raw OCR  : {raw}")
    print(f"     Resolved : {resolved_str} (Conf: {conf:.2f}, Gate: {gate})\n")
